# 04 · 物理仿真与同步多模态数据
执行模块三的 cabinet2 轨迹，使用同一 MuJoCo 状态生成双相机 RGB、深度、部件 Mask、点云、位姿和帧间事件计数，以及关节、接触和动作序列。

随机化在每个 episode 的种子下重新采样。事件是亮度阈值模拟，物性仍为明确声明的模型先验；这些数据不标为真机采集。

In [ ]:
import json
import os
import sys
import uuid
from pathlib import Path

os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("OMP_NUM_THREADS", "4")
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/physrealx").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if Path(sys.prefix).name != "PhysReal-X":
    raise RuntimeError("请选择 PhysReal-X Conda 环境的 Jupyter 内核")
from examples._common import (
    NotebookRun,
    operation_receipt,
    output_artifact,
    require_completed,
)

from physrealx import ArtifactRef, PhysRealX, VersionedRef

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
OUTPUT_DIR = PROJECT_ROOT / "examples/simulation_and_data_generation/output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run = NotebookRun(
    OUTPUT_DIR / "report.json",
    ("input", "sensors", "expert", "failure", "quality", "preview", "appearance", "export"),
    {"python": sys.executable, "conda_prefix": sys.prefix},
    evidence_scope="mujoco_multimodal_simulator_episodes",
)
run_key = uuid.uuid4().hex[:12]
print(sys.executable)

In [ ]:
with run.stage("input") as stage:
    import numpy as np
    from scipy.spatial.transform import Rotation

    from physrealx import MethodCapability, RunSpec, SignalSpec

    upstream = require_completed(
        PROJECT_ROOT / "examples/scene_and_task_generation/output/report.json"
    )
    previous = upstream["stages"]["export"]
    scene_ref = VersionedRef.model_validate(previous["scene"])
    original_task = px.tasks.get(VersionedRef.model_validate(previous["task"]))
    scene = px.scenes.get(scene_ref)
    plan = json.loads(px.artifacts.get_bytes(ArtifactRef.model_validate(previous["trajectory"])))
    stage["scene"] = scene_ref

In [ ]:
with run.stage("sensors") as stage:
    target = np.array(scene.entities[0].pose.translation_m) + np.array([0, 0, 0.17])
    cameras = {}
    for name, offset in {"front": [0.85, -0.65, 0.5], "side": [0.6, 0.7, 0.4]}.items():
        position = target + offset
        backward = position - target
        backward /= np.linalg.norm(backward)
        right = np.cross([0, 0, 1], backward)
        right /= np.linalg.norm(right)
        up = np.cross(backward, right)
        cameras[name] = {
            "pose": {
                "translation_m": position.tolist(),
                "rotation_xyzw": Rotation.from_matrix(np.column_stack([right, up, backward]))
                .as_quat()
                .tolist(),
            },
            "width": 64,
            "height": 48,
            "max_depth_m": 5.0,
            "event_threshold": 0.2,
        }
    observations = list(original_task.observations)
    for name in cameras:
        for modality, shape, dtype, unit in (
            ("rgb", (48, 64, 3), "uint8", "1"),
            ("depth", (48, 64), "float32", "m"),
            ("mask", (48, 64), "int32", "1"),
            ("events", (48, 64, 2), "int32", "1"),
            ("pointcloud", (48, 64, 3), "float32", "m"),
            ("intrinsics", (3, 3), "float64", "1"),
            ("position", (3,), "float64", "m"),
            ("orientation", (4,), "float64", "1"),
        ):
            observations.append(
                SignalSpec(
                    name=f"camera.{name}.{modality}",
                    shape=shape,
                    dtype=dtype,
                    unit=unit,
                    frame="world" if modality in {"position", "orientation"} else f"camera.{name}",
                    frequency_hz=20,
                )
            )
    link_count = sum(len(px.assets.get(entity.asset).links) for entity in scene.entities)
    for name, shape, unit in (
        ("body.position", (link_count, 3), "m"),
        ("body.orientation", (link_count, 4), "1"),
        ("body.external_force", (link_count, 3), "N"),
        ("body.external_torque", (link_count, 3), "N*m"),
        ("body.wrench_origin", (link_count, 3), "m"),
    ):
        observations.append(
            SignalSpec(
                name=name, shape=shape, dtype="float32", unit=unit, frame="world", frequency_hz=20
            )
        )
    task = original_task.model_copy(
        update={
            "resource_id": f"cabinet2.rendered.{run_key}",
            "observations": tuple(observations),
            "max_steps": 160,
        }
    )
    task_ref = px.tasks.register(task)
    backend_config = dict(previous["backend_config"])
    backend_config.update(
        cameras=cameras,
        randomization={
            "mass_scale": [0.8, 1.2],
            "damping_scale": [0.8, 1.2],
            "friction_scale": [0.8, 1.2],
            "initial_position_jitter": 0.005,
            "color_jitter": 0.15,
            "light_jitter": 0.15,
            "camera_position_jitter_m": 0.02,
            "occlusion_probability": 0.4,
            "occlusion_fraction": [0.08, 0.18],
        },
    )
    backend = px.methods.bind(MethodCapability.SIMULATION_RUN, "mujoco", config=backend_config)
    stage["task"] = task_ref
    stage["backend"] = backend
    print("同步信号:", len(observations), "；相机:", list(cameras))

In [ ]:
with run.stage("expert") as stage:
    expert_spec = RunSpec(
        run_id=str(uuid.uuid4()),
        scene=scene_ref,
        task=task_ref,
        backend=backend,
        policy=px.methods.bind(
            MethodCapability.POLICY_INFERENCE,
            "joint_sequence",
            config={"commands": plan["commands"]},
        ),
        episode_count=20,
        stop_on_success=False,
        base_seed=20260,
        dataset_id=f"cabinet2.expert.{run_key}",
        dataset_version="1.0.0",
        record_signals=tuple(signal.name for signal in task.observations),
        tags=("expert_demonstration",),
    )
    expert_run = px.runs.submit(expert_spec)
    expert_result = px.runs.wait(expert_run)
    expert_episodes = px.runs.episodes(expert_run)
    assert all(episode.success for episode in expert_episodes)
    stage["run"] = expert_run
    stage["dataset"] = expert_result.dataset
    print("成功轨迹:", len(expert_episodes), "帧数:", sum(len(ep.steps) for ep in expert_episodes))

In [ ]:
with run.stage("failure") as stage:
    failure_spec = expert_spec.model_copy(
        update={
            "run_id": str(uuid.uuid4()),
            "dataset_id": f"cabinet2.failure.{run_key}",
            "base_seed": 30260,
            "episode_count": 6,
            "policy": px.methods.bind(
                MethodCapability.POLICY_INFERENCE, "joint_target", config={"target": [0.0]}
            ),
        }
    )
    failure_run = px.runs.submit(failure_spec)
    failure_result = px.runs.wait(failure_run)
    failed_episodes = px.runs.episodes(failure_run)
    assert not any(episode.success for episode in failed_episodes)
    stage["run"] = failure_run
    stage["dataset"] = failure_result.dataset
    print("未达目标的真实仿真轨迹:", len(failed_episodes))

## 外部执行器与被动抽屉接触
直线执行器使用标准 AssetSpec 和显式物性先验。只有推杆通电驱动，抽屉没有电机；成功判据读取被动抽屉位置，接触力必须非零。这是直线执行器推压演示；多指抓取和现场机械臂仍需相应机器人资产与部署接口。


In [ ]:
with run.stage("quality") as stage:
    from physrealx.integrations.datasets import EpisodeCollection, validate_dataset

    assert backend.config["controlled_joints"] == ["pusher/extension"]
    peak_force = max(
        s.observations["contact.force"].inline[0] for ep in expert_episodes for s in ep.steps
    )
    assert 0 < peak_force < 100, "被动抽屉必须由机器人实际接触驱动"
    for reference in (expert_result.dataset, failure_result.dataset):
        quality = validate_dataset(px.artifacts, px.datasets.get(reference))
        print(quality)
    collection = px.artifacts.put_json(
        EpisodeCollection(datasets=(expert_result.dataset, failure_result.dataset))
    )
    stage["collection"] = collection
    stage["peak_contact_force_n"] = peak_force
    stage["control_scope"] = "robot pusher only; object joints passive"

In [ ]:
with run.stage("preview") as stage:
    import matplotlib.pyplot as plt

    from physrealx.integrations._artifacts import read_array

    sample = expert_episodes[0].steps[len(expert_episodes[0].steps) // 2]
    figure, axes = plt.subplots(2, 4, figsize=(12, 5))
    for row, name in enumerate(cameras):
        for column, modality in enumerate(("rgb", "depth", "mask", "events")):
            array = read_array(
                px.artifacts, sample.observations[f"camera.{name}.{modality}"].artifact
            )
            if modality == "events":
                array = array[:, :, 0] - array[:, :, 1]
            axes[row, column].imshow(array, cmap="viridis" if modality != "rgb" else None)
            axes[row, column].set_title(f"{name} / {modality}")
            axes[row, column].axis("off")
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "multimodal.png", dpi=140)
    plt.show()
    stage["preview"] = str(OUTPUT_DIR / "multimodal.png")

## 物理状态驱动的已配准高斯外观

读取同一物理资产绑定的 ArtGS 模型，以实际记录的被动抽屉位置驱动外观，摄像机经过完整位姿变换。此视图单独显示 cabinet2；双相机全场景、碰撞和多模态数据仍由前面的 MuJoCo 记录提供。超出实拍起止状态的部分按 USD 关节先验外推并标明。

In [ ]:
with run.stage("appearance") as stage:
    from physrealx.integrations.gaussian_render import RegisteredRenderInput
    from physrealx.integrations.sensors import CameraSpec

    cabinet = px.assets.get(
        next(entity.asset for entity in scene.entities if entity.name == "cabinet")
    )
    registration_ref = ArtifactRef.model_validate(cabinet.metadata["registered_appearance"])
    view = dict(cameras["front"])
    view.update(width=320, height=240)
    trajectory = px.artifacts.put_json(
        RegisteredRenderInput(
            registration=registration_ref,
            episode=px.datasets.get(expert_result.dataset).episodes[0],
            entity="cabinet",
            camera=CameraSpec.model_validate(view),
            frame_indices=(0, 40, 80, 120, 160),
        )
    )
    operation = px.renders.generate(trajectory, method="registered_artgs")
    assert operation.metrics["physical_motion_span_m"] > 0.1
    assert operation.metrics["foreground_pixels"] > 1000
    streams_ref = output_artifact(operation, "render_streams")
    streams = json.loads(px.artifacts.get_bytes(streams_ref))
    images = read_array(px.artifacts, ArtifactRef.model_validate(streams["streams"]["rgb"]))
    fig, axes = plt.subplots(1, len(images), figsize=(15, 3))
    for axis, pixels, position in zip(axes, images, streams["recorded_joint_positions"]):
        axis.imshow(pixels)
        axis.set_title(f"Recorded q = {position:.3f} m")
        axis.axis("off")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "registered-appearance.png", dpi=150)
    plt.show()
    stage["operation"] = operation_receipt(operation)
    stage["streams"] = streams_ref
    stage["outside_observed_span"] = streams["outside_observed_span"]

In [ ]:
with run.stage("export") as stage:
    stage["scene"] = scene_ref
    stage["task"] = task_ref
    stage["expert_dataset"] = expert_result.dataset
    stage["failure_dataset"] = failure_result.dataset
    stage["episode_collection"] = collection
    stage["evaluation_run_spec"] = expert_spec
report = run.complete()
print(report["execution_status"], OUTPUT_DIR / "report.json")